# Ejercicio guiado — Palmer Penguins (calidad de datos e imputación)

**Asignatura:** Fundamentos de aprendizaje automático  
**Duración orientativa:** 1,5–2 horas  

## Objetivo

Practicar el **mismo flujo** que viste en los notebooks de la unidad:

- **EDA inicial:** forma de la tabla, tipos, variable objetivo.
- **Calidad:** contar `NaN`, localizar patrones, distinguir outliers de faltantes.
- **Partición train/test** sin filtrar información del conjunto de prueba.
- **Imputación** con `sklearn` (`fit` solo en entrenamiento).

**Dataset:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) vía `seaborn` (`sns.load_dataset("penguins")`). Cada fila es un pingüino; la variable objetivo será **`species`** (especie: Adelie, Chinstrap, Gentoo).

## Antes de empezar

1. Entorno: `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`.
2. Ejecuta las celdas **en orden**.
3. Las secciones marcadas como **Tu trabajo** deben completarse por ti (código y/o breves comentarios en markdown si el profesor lo pide).

**Referencias en el repositorio:**

- Ideas de EDA y lectura de variables: notebook de Churn (guía / guiado).
- Faltantes, IQR, `SimpleImputer`, `KNNImputer`, `ColumnTransformer`: notebook *Imputación y calidad — Titanic*.

## Diccionario breve

| Columna | Tipo usual | Descripción |
|---------|------------|-------------|
| **`species`** | categórica | **Objetivo:** especie (`Adelie`, `Chinstrap`, `Gentoo`) |
| `island` | categórica | Isla de observación |
| `bill_length_mm` | numérica | Largo del pico (mm) |
| `bill_depth_mm` | numérica | Profundidad del pico (mm) |
| `flipper_length_mm` | numérica | Largo de la aleta (mm) |
| `body_mass_g` | numérica | Masa corporal (g) |
| `sex` | categórica | `Male` / `Female` (puede tener faltantes) |

En la versión típica del dataset en Seaborn hay **unos pocos** faltantes en medidas y algo más en `sex`; lo comprobarás con datos reales en tu ejecución.

---
## 1. Carga y objetivo del análisis

**Tu trabajo:**

1. Carga el dataset con `sns.load_dataset("penguins")`.
2. Imprime `shape` y muestra las primeras filas (`head`).
3. En una línea (comentario o celda markdown): ¿cuántas filas y columnas obtienes? ¿La variable objetivo es equilibrada entre especies? (usa `value_counts` normalizado o no, según prefieras).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

# TODO: df = sns.load_dataset(...)
# TODO: print(shape), head(), value_counts de species



## 2. Tipos y primer diagnóstico

**Tu trabajo:**

1. `info()` o `dtypes` para ver tipos.
2. Al menos **un gráfico** exploratorio que ayude a ver la separación de especies (por ejemplo `sns.scatterplot` con dos medidas del pico/aleta, coloreando por `species`, o `pairplot` en un subconjunto de columnas si el equipo lo permite).
3. Opcional: histograma o KDE de una variable numérica por especie.

In [ ]:
# TODO: dtypes / info

# TODO: un gráfico exploratorio (scatter o similar)



## 3. Valores faltantes

**Tu trabajo:**

1. Suma de `NaN` por columna (`isna().sum()`), ordenada de mayor a menor si hay varios.
2. Porcentaje sobre el total de filas.
3. Opcional: mapa de calor de faltantes en un subconjunto de filas (como en el notebook Titanic), o comprobar si las filas con NaN en medidas coinciden entre columnas.
4. **Pregunta:** ¿eliminarías filas, imputarías todo, o mezcla? Justifica en una o dos frases (no hay una única respuesta correcta; importa el argumento).

In [ ]:
# TODO: tabla n_missing y % por columna

# TODO (opcional): heatmap df.isna().iloc[:...]



## 4. Outliers vs faltantes (regla IQR)

**Tu trabajo:**

1. Elige **dos** variables numéricas (por ejemplo `bill_length_mm` y `body_mass_g`).
2. Para cada una, calcula límites IQR con factor 1,5 y cuenta cuántas filas quedarían como *candidatas* a outlier (solo entre valores no nulos).
3. Opcional: `sns.boxplot` para esas variables.
4. **Importante:** un outlier es un valor **observado** extremo; no es lo mismo que `NaN`. Una frase que lo aclare en tu informe.

In [ ]:
def iqr_bounds(s: pd.Series, k=1.5):
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

# TODO: aplicar a dos columnas e imprimir límites y número de candidatos a outlier



## 5. Partición entrenamiento / prueba

**Tu trabajo:**

1. Define `y = species` y `X` con las columnas predictoras que quieras usar (por ejemplo todas excepto `species`).
2. Usa `train_test_split` con `test_size=0.25`, `random_state=RANDOM_STATE` y **`stratify=y`** (clasificación con tres clases).
3. Imprime tamaños de train y test y comprueba que en train la proporción de especies es razonablemente similar a la del dataset completo.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: feature_cols, X, y, train_test_split con stratify=y



## 6. Imputación con `sklearn` (solo entrena en train)

**Tu trabajo:**

1. Separa nombres de columnas **numéricas** y **categóricas** de `X`.
2. Construye un `ColumnTransformer` con:
   - `Pipeline` numérico: `SimpleImputer(strategy="median")` (o la estrategia que defiendas).
   - `Pipeline` categórico: `SimpleImputer(strategy="most_frequent")` para `sex` e `island`.
3. Haz `fit` en **`X_train`** y `transform` en train y test.
4. Convierte el resultado a `DataFrame` con los nombres de columnas correctos y verifica que **no queden NaN** en las columnas transformadas.

**Opcional (avanzado):** en las numéricas, encadena `StandardScaler` + `KNNImputer` como en el notebook Titanic y compara con la imputación simple (solo si ya dominas el flujo).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# TODO: num_cols, cat_cols
# TODO: ColumnTransformer + fit en X_train + transform train/test
# TODO: DataFrame final y comprobación isna().sum()



## 7. Cierre (entrega)

Responde de forma breve (puedes añadir una celda markdown abajo):

1. ¿Qué columnas tenían faltantes y en qué magnitud?
2. ¿Los “outliers” por IQR te parecen errores de medición o valores plausibles?
3. ¿Por qué el `fit` del imputador debe hacerse solo con `X_train`?
4. ¿Qué harías como siguiente paso antes de un clasificador (`LogisticRegression`, etc.)? (pista: codificación de categóricas, `Pipeline` completo.)

---

*Fin del ejercicio guiado.*